# Sumi-e style LoRA for LTX-2.5 (Colab)

Trains a **video-branch style LoRA** on [takara-ai/Sumi-e_no_kurozuappu_ink_painting_closeups](https://huggingface.co/datasets/takara-ai/Sumi-e_no_kurozuappu_ink_painting_closeups)
(1,250 synthetic sumi-e images, CC BY 4.0) using Lightricks' official trainer (`packages/ltx-trainer` in [Lightricks/LTX-2](https://github.com/Lightricks/LTX-2)).

**Before you start**
1. Runtime → Change runtime type → **A100** (40 or 80 GB) or **H100**. L4/T4 don't have enough memory for the 22B model.
2. On Hugging Face, open [Lightricks/LTX-2.5](https://huggingface.co/Lightricks/LTX-2.5) and accept the model terms.
3. Add a Hugging Face **Read** token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar), with notebook access on.

**Credits:** a full run (setup, ~70 GB model download, captioning, preprocessing, 2,000 steps, validation videos) takes
roughly 3–5 hours of A100 time. **Disconnect the runtime as soon as training finishes**: an idle GPU still uses credits.
Data, the latent cache and checkpoints are saved to Google Drive, so a disconnect doesn't lose finished work.

## 1. Settings and GPU check

In [ ]:
import subprocess, os
WORK = "/content/drive/MyDrive/sumi_e_ltx25"   # data, cache, checkpoints (persist across disconnects)
MODELS = "/content/models/ltx-2.5"              # 70 GB of weights: local disk only
REPO_URL = "https://github.com/enoky/Sumi-e_LTX_2.5_LORA.git"   # this repo (scripts/). If it is private, upload scripts/ to /content/sumi/scripts instead
REPO_BRANCH = "main"
LTX_COMMIT = "f0f2f8607189c78a72420b2b63cb1975b5eca179"          # LTX-2 version these scripts were written against

TARGET_IMAGES = 300     # curated subset size
STEPS = 2000
RANK = 32
RESOLUTION = 1024       # square training bucket; drop to 768 if the dataset images are smaller
TRIGGER = "sumi-e ink painting,"
CAPTION_MODE = "vlm"    # "vlm" = Qwen2.5-VL captions (better, ~10 min); "prompt" = cleaned SD prompts (no GPU)

mem_gb = int(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]).decode().split()[0]) / 1024
name = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode().strip()
if mem_gb < 38:
    raise SystemExit(f"{name} has {mem_gb:.0f} GB. Switch the runtime to A100 or H100.")
GPU = "h100" if "H100" in name else ("a100-80" if mem_gb > 70 else "a100-40")
print(f"{name}, {mem_gb:.0f} GB -> profile {GPU}")
!df -h /content | tail -1

## 2. Google Drive and Hugging Face login

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
os.makedirs(WORK, exist_ok=True)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!pip -q install -U huggingface_hub datasets   # a cudf/pyarrow conflict warning here is harmless
!hf auth whoami

## 3. Start the model download (runs in the background)

Downloads the **dev** (full) transformer, which is the one to train LoRAs on, plus the LTX Gemma 4 text encoder and
the lighter convolutional video VAE, plus the small audio VAE that the test script (section 12) needs. About 70 GB. It keeps going while you run the data-prep cells below.

In [ ]:
os.makedirs(MODELS, exist_ok=True)
dl = subprocess.Popen(
    ["hf", "download", "Lightricks/LTX-2.5",
     "diffusion_models/ltx-2.5-22b-dev-transformer-bf16.safetensors",
     "text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors",
     "vae/ltx-2.5-video-vae-conv-bf16.safetensors",
     "vae/ltx-2.5-audio-vae-bf16.safetensors",
     "--local-dir", MODELS],
    stdout=open("/content/model_download.log", "w"), stderr=subprocess.STDOUT)
print("Download started; log at /content/model_download.log")

## 4. Get the scripts and install the LTX trainer

In [ ]:
%cd /content
!rm -rf sumi && git clone -q -b {REPO_BRANCH} {REPO_URL} sumi
!rm -rf LTX-2 && git clone -q https://github.com/Lightricks/LTX-2 && cd LTX-2 && git checkout -q {LTX_COMMIT}
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!cd /content/LTX-2 && uv sync 2>&1 | tail -3
!cd /content/LTX-2 && uv run python -c "import torch; print('trainer env torch', torch.__version__, 'CUDA ok:', torch.cuda.is_available())"

If the last line says `CUDA ok: False`, Colab's NVIDIA driver is older than the CUDA build the trainer pins.
Check `nvidia-smi` for the driver's CUDA version and see the troubleshooting note in the repo README.

## 5. Download the dataset

In [ ]:
!python /content/sumi/scripts/download_dataset.py --out {WORK}/data/raw

## 6. Curate a diverse subset

Removes near-duplicates and picks `TARGET_IMAGES` images that cover the range of subjects. Look over the contact sheets.
To drop bad images (fake seals or text, muddy blobs, anything that doesn't look like ink), list their filenames, one
per line, in `{WORK}/exclude.txt` and run this cell again.

In [ ]:
!python /content/sumi/scripts/curate.py --raw {WORK}/data/raw --out {WORK}/data/curated --target {TARGET_IMAGES} --exclude {WORK}/exclude.txt
from IPython.display import Image as IPImage, display
import glob
for p in sorted(glob.glob(f"{WORK}/data/curated/contact_sheet_*.jpg")):
    print(p); display(IPImage(p, width=1100))

## 7. Caption the images

In [ ]:
!python /content/sumi/scripts/caption.py --data {WORK}/data/curated --mode {CAPTION_MODE}
import torch, gc; gc.collect(); torch.cuda.empty_cache()

Open `{WORK}/data/curated/dataset.json` and fix any obviously wrong captions before you continue. Don't add the trigger word; the trainer adds it.

## 8. Wait for the model download and write the training config

In [ ]:
print("waiting for model download..."); rc = dl.wait()
!tail -3 /content/model_download.log
assert rc == 0, "model download failed: check the log (did you accept the LTX-2.5 terms?)"
!du -sh {MODELS}/*
CFG = f"{WORK}/sumi_e_lora.yaml"
!python /content/sumi/scripts/make_config.py --models {MODELS} --data {WORK}/data/curated --out {CFG} \
    --output-dir {WORK}/outputs/sumi_e_lora --gpu {GPU} --steps {STEPS} --rank {RANK} \
    --resolution {RESOLUTION} --trigger "{TRIGGER}"

## 9. Preprocess (encode images to latents, captions to text embeddings)

About 10–20 minutes. The cache is written to Drive, so after a disconnect this step skips work that's already done.
`decode: false` by default; to check the crops, set `training_strategy.data.decode: true` in the YAML and run again with `--overwrite`.

In [ ]:
!cd /content/LTX-2/packages/ltx-trainer && MPLBACKEND=Agg uv run python scripts/preprocess.py {CFG}

## 10. Train

Prints loss as it goes. Validation videos are written every 500 steps and checkpoints every 250 steps, to
`{WORK}/outputs/sumi_e_lora`. **To resume after a disconnect:** run cells 1–4 and 8, set
`checkpoints.load_checkpoint` in the YAML to `{WORK}/outputs/sumi_e_lora`, and run this cell.

Optional: before the full run, set `STEPS = 50`, rerun cell 8, and time a short run (s/step × 2000 = full run time).

In [ ]:
!cd /content/LTX-2/packages/ltx-trainer && MPLBACKEND=Agg uv run python scripts/train.py {CFG}

## 11. Look at the results

In [ ]:
!ls -R {WORK}/outputs/sumi_e_lora | head -50
from IPython.display import Video
vids = sorted(glob.glob(f"{WORK}/outputs/sumi_e_lora/**/*.mp4", recursive=True))
for v in vids[-4:]:
    print(v); display(Video(v, embed=True, width=384))

## 12. Test the LoRA against the base model

Renders the same prompts with the same seed on the base model (strength 0) and with the LoRA, then writes
`{WORK}/tests/index.html`, a side-by-side grid (download it and open it in a browser). The default prompts cover
close-ups like the training data, ink-style motion, wider shots and new subjects, and one prompt **without** the trigger
to check that the style doesn't leak into everything.

**In a new Colab session**, run cells 1–4 first; this cell waits for the model download.

Cost: each video takes roughly 1–3 min on an H100 and longer on an A100 40GB (weights stream from CPU RAM there).
The defaults make 12 videos. To compare checkpoints, add e.g. `--checkpoints step_01500 step_02000`; to test strengths,
use `--strengths 0 0.7 1.0`. Pass `--prompts my_prompts.txt` to use your own prompts (one per line, `{t}` = trigger).

In [ ]:
if "dl" in globals() and dl.poll() is None:
    print("waiting for model download..."); dl.wait()
!cd /content/LTX-2 && MPLBACKEND=Agg uv run python /content/sumi/scripts/test_lora.py --models {MODELS} \
    --lora {WORK}/outputs/sumi_e_lora --strengths 0 1.0 --steps 30 --out {WORK}/tests
from IPython.display import Video, display
import glob
for v in sorted(glob.glob(f"{WORK}/tests/*.mp4")):
    print(os.path.basename(v)); display(Video(v, embed=True, width=384))

## 13. Upload the LoRA to Hugging Face

Uploads one checkpoint (renamed `sumi-e_ltx-2.5_lora.safetensors`), a model card with the trigger word, usage,
and the required license and dataset credit, plus up to 6 LoRA renders from section 12.

1. Create a token with **Write** access at huggingface.co/settings/tokens and add it as a Colab secret named
   `HF_WRITE_TOKEN`. The `HF_TOKEN` secret from section 2 is read-only.
2. Set `HF_REPO` and `CHECKPOINT` below. Pick the checkpoint that looked best in the tests; `None` uploads the latest.
3. The repo is created **private**. Make it public on Hugging Face when you're happy with it (or set `PUBLIC = True`).

In a new session, run cells 1 and 2 first.

In [ ]:
HF_REPO = "your-username/sumi-e-ltx-2.5-lora"   # <- change
CHECKPOINT = None                               # e.g. "step_01500"; None = latest
PUBLIC = False

from google.colab import userdata
!git -C /content/sumi pull -q 2>/dev/null || git clone -q -b {REPO_BRANCH} {REPO_URL} /content/sumi
ckpt_flag = f"--checkpoint {CHECKPOINT}" if CHECKPOINT else ""
public_flag = "--public" if PUBLIC else ""
samples_flag = f"--samples {WORK}/tests" if os.path.isdir(f"{WORK}/tests") else ""
os.environ["HF_UPLOAD_TOKEN"] = userdata.get("HF_WRITE_TOKEN")
!HF_TOKEN=$HF_UPLOAD_TOKEN python /content/sumi/scripts/upload_to_hf.py --repo {HF_REPO} \
    --lora {WORK}/outputs/sumi_e_lora {ckpt_flag} {samples_flag} {public_flag} --trigger "{TRIGGER}"

**Done? Disconnect now:** Runtime → Disconnect and delete runtime (after training, or after testing).

The LoRA is `{WORK}/outputs/sumi_e_lora/checkpoints/step_NNNNN/weights.safetensors`. Use it with any
LTX-2.5 pipeline via `--lora <path> 1.0`, and start prompts with the trigger `sumi-e ink painting,`. Credit the dataset
(CC BY 4.0, Takara.ai) when you share the LoRA.